# MarketAI Whisper API Server

This notebook runs a Whisper API server that MarketAI can connect to for speech-to-text.

**Setup:**
1. Runtime > Change runtime type > T4 GPU > Save
2. Run all cells
3. Copy the ngrok URL and paste it into MarketAI settings

**Keep this tab open while using MarketAI voice features.**

In [ ]:
# Step 1: Install dependencies
!pip install -q openai-whisper flask pyngrok

In [ ]:
# Step 2: Setup ngrok auth token
# Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # <-- Paste your ngrok auth token here

if not NGROK_AUTH_TOKEN:
    print("\u26a0\ufe0f Please set NGROK_AUTH_TOKEN above!")
    print("Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("\u2705 ngrok token set!")

In [ ]:
# Step 3: Load Whisper model (takes ~2 min first time)
import whisper
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("\u23f3 Loading large-v3 model...")
model = whisper.load_model("large-v3")
print("\u2705 Model loaded!")

In [ ]:
# Step 4: Start API Server
import os
import json
import tempfile
from flask import Flask, request, jsonify
from pyngrok import ngrok
from threading import Thread

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 200 * 1024 * 1024  # 200MB

@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model': 'large-v3',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    })

@app.route('/transcribe', methods=['POST'])
def transcribe():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded'}), 400

    file = request.files['file']
    initial_prompt = request.form.get('initial_prompt', '')
    language = request.form.get('language', 'zh')

    # Save to temp file
    with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(file.filename)[1]) as tmp:
        file.save(tmp.name)
        tmp_path = tmp.name

    try:
        opts = {'language': language}
        if initial_prompt:
            opts['initial_prompt'] = initial_prompt

        result = model.transcribe(tmp_path, **opts)

        segments = []
        timestamped_lines = []
        for seg in result.get('segments', []):
            start = int(seg['start'])
            mm, ss = divmod(start, 60)
            text = seg['text'].strip()
            segments.append({
                'start': seg['start'],
                'end': seg['end'],
                'text': text
            })
            timestamped_lines.append(f'[{mm:02d}:{ss:02d}] {text}')

        return jsonify({
            'text': result['text'],
            'timestamped': '\n'.join(timestamped_lines),
            'segments': segments
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500
    finally:
        os.unlink(tmp_path)

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print("=" * 60)
print(f"\U0001f680 Whisper API is running!")
print(f"")
print(f"\U0001f517 API URL: {public_url}")
print(f"")
print(f"Copy this URL and paste it into MarketAI settings.")
print("=" * 60)

# Run Flask in background thread
Thread(target=lambda: app.run(port=5000)).start()

In [ ]:
# (Optional) Test the API
import requests
r = requests.get(f"{public_url}/health")
print(r.json())